# train_run — một lượt huấn luyện (tuần 2)
1. Sửa `MODEL`, `SEQ_LEN`, `SEED` ở cell 1. Tên lượt tự đặt `<model>_L<len>_s<seed>`.
2. Settings: GPU T4 · Internet On · Environment **Pin to original environment (2026-11-19)**.
3. Add Input: Dataset `mamba-btl-data` và `mamba-btl-wheels`. Chạy tiếp lượt bị ngắt: Add Input thêm output (version) trước của chính notebook này.
4. Bấm **Save Version → Save & Run All** (chạy nền, tối đa 12 giờ). Ghi lượt vào Google Sheet trước khi chạy.
5. Xong: điền Giờ GPU, Val BPC, ký tự/s vào Sheet; tải `logs/<run_name>.csv` và `_summary.json` để đưa vào repo.

In [ ]:
# 1. Cấu hình lượt chạy — CHỈ SỬA Ở ĐÂY
MODEL = "mamba"        # "mamba" | "transformer"
SEQ_LEN = 256          # 256 | 1024 | 4096
SEED = 0               # 0; seed phụ 1, 2 chỉ ở L = 1024
EXTRA = ""             # ghi đè thêm nếu cần, vd "--set train.train_chars=16_384_000" khi chạy thử
RUN_NAME = f"{MODEL}_L{SEQ_LEN}_s{SEED}"
print(RUN_NAME)

In [ ]:
# 2. Clone repo + cài mamba-ssm từ wheel
%cd /kaggle/working
!rm -rf mamba-vi-keyboard && git clone -q https://github.com/daisyngocha/mamba-vi-keyboard.git
%cd /kaggle/working/mamba-vi-keyboard
!git log --oneline -1
from utils import kaggle_input
wheel_dir = kaggle_input("mamba-btl-wheels")
wheels = sorted(str(p) for p in wheel_dir.rglob("*.whl")) if wheel_dir else []
assert wheels or MODEL != "mamba", "Chưa Add Input Dataset mamba-btl-wheels"
if wheels:
    !pip install -q --no-deps {" ".join(wheels)}
!pip install -q pyyaml
assert kaggle_input("mamba-btl-data"), "Chưa Add Input Dataset mamba-btl-data"

In [ ]:
# 3. Kiểm tra nhanh trước khi chạy dài
import torch, sys
print(torch.__version__, torch.version.cuda, sys.version.split()[0], torch.cuda.get_device_name())
if MODEL == "mamba":
    from mamba_ssm import Mamba
    print("mamba-ssm OK")
!bash scripts/smoke_test.sh

In [ ]:
# 4. Huấn luyện (tự resume nếu output version trước đã được Add Input)
!python train.py --config configs/{MODEL}.yaml --set data.seq_len={SEQ_LEN} --set seed={SEED} {EXTRA}

In [ ]:
# 5. Tóm tắt để điền vào Google Sheet
import json, pandas as pd
s = json.load(open(f"/kaggle/working/logs/{RUN_NAME}_summary.json"))
print({k: v for k, v in s.items() if k != "config"})
print(f"Giờ GPU (huấn luyện): {s['train_time_s']/3600:.2f}")
log = pd.read_csv(f"/kaggle/working/logs/{RUN_NAME}.csv")
ax = log.dropna(subset=["val_bpc"]).plot(x="step", y="val_bpc", title=RUN_NAME, figsize=(7, 3.5))
ax.figure.savefig(f"/kaggle/working/logs/{RUN_NAME}_val_bpc.png", dpi=120)